# MSE 433 Report Figures

Publication-quality figures for the Household Energy Cost & Usage Dashboard final report.

**Colour palette (Section 7.8 of design doc)**
- Navy `#1B2A4A` — text, axes, baselines
- Blue `#2E75B6` — XGBoost, primary data
- Orange `#E8792F` — Ridge, secondary
- Red `#D32F2F` — Seasonal Naive, alerts
- Green `#388E3C` — success, improvements

All figures saved to `backend/results/figures/` at 300 DPI.

## 0. Setup and Imports

In [ ]:
import sys
import pathlib
import json
import warnings
warnings.filterwarnings('ignore')

# Resolve project root (backend/)
BACKEND = pathlib.Path('/Users/jeevanparmar/Uni/433/Final_Study/433-Final-Project/backend')
sys.path.insert(0, str(BACKEND))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import joblib

# Project palette
NAVY   = '#1B2A4A'
BLUE   = '#2E75B6'
ORANGE = '#E8792F'
RED    = '#D32F2F'
GREEN  = '#388E3C'
GREY   = '#F7F8FA'

MODEL_COLOURS = {
    'Seasonal Naive': RED,
    'Ridge':          ORANGE,
    'XGBoost':        BLUE,
}

# Matplotlib global defaults for publication quality
matplotlib.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.edgecolor':   NAVY,
    'axes.linewidth':   1.2,
    'axes.labelcolor':  NAVY,
    'axes.labelsize':   13,
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
    'axes.titlecolor':  NAVY,
    'xtick.color':      NAVY,
    'ytick.color':      NAVY,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '#CCCCCC',
    'grid.color':       '#E0E0E0',
    'grid.linewidth':   0.8,
    'lines.linewidth':  2.0,
    'font.family':      'DejaVu Sans',
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'savefig.facecolor': 'white',
})

FIGURES_DIR = BACKEND / 'results' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Backend root : {BACKEND}')
print(f'Figures dir  : {FIGURES_DIR}')

## 1. Load All Artefacts

In [ ]:
# ── Test metrics ────────────────────────────────────────────────────────────
test_metrics = pd.read_csv(BACKEND / 'results' / 'test_metrics.csv')
val_metrics  = pd.read_csv(BACKEND / 'results' / 'validation_metrics.csv')

# Pretty-print model names
NAME_MAP = {
    'seasonal_naive': 'Seasonal Naive',
    'ridge':          'Ridge',
    'xgboost':        'XGBoost',
}
test_metrics['model_label'] = test_metrics['model'].map(NAME_MAP)
val_metrics['model_label']  = val_metrics['model'].map(NAME_MAP)

print('Test metrics:')
print(test_metrics[['model_label', 'mae', 'rmse', 'mape', 'peak_mae']].to_string(index=False))

# ── Hourly clean data ────────────────────────────────────────────────────────
hourly = pd.read_parquet(BACKEND / 'data' / 'processed' / 'hourly_clean.parquet')
hourly.index = pd.to_datetime(hourly.index)
print(f'\nHourly data: {hourly.shape}, {hourly.index.min()} to {hourly.index.max()}')

# ── Feature matrix ───────────────────────────────────────────────────────────
features = pd.read_parquet(BACKEND / 'data' / 'features' / 'features.parquet')
features.index = pd.to_datetime(features.index)
print(f'Features:    {features.shape}')

# ── Conformal widths ─────────────────────────────────────────────────────────
with open(BACKEND / 'models' / 'conformal_widths.json') as f:
    conformal_widths = json.load(f)
print(f'Conformal widths loaded for {len(conformal_widths)} horizons')

# ── SHAP values ──────────────────────────────────────────────────────────────
shap_df = pd.read_parquet(BACKEND / 'results' / 'shap_values.parquet')
shap_df.index = pd.to_datetime(shap_df.index)
print(f'SHAP values: {shap_df.shape}')

# ── Temporal splits ──────────────────────────────────────────────────────────
TRAIN_END = pd.Timestamp('2008-12-31')
VAL_END   = pd.Timestamp('2010-06-30')

train_feat = features[features.index <= TRAIN_END]
val_feat   = features[(features.index > TRAIN_END) & (features.index <= VAL_END)]
test_feat  = features[features.index > VAL_END]

TARGET = 'Global_active_power'
print(f'\nSplit sizes  — train: {len(train_feat)}, val: {len(val_feat)}, test: {len(test_feat)}')

## 2. Figure 1 — Model Comparison Bar Chart

Grouped bar chart comparing MAE, RMSE, and MAPE across the three models on the held-out test set.  
XGBoost is the primary model (blue); Ridge provides the best overall MAE (orange);  
Seasonal Naive is the reference baseline (red).

In [ ]:
# ── Data preparation ─────────────────────────────────────────────────────────
metrics_plot = test_metrics.set_index('model_label')
model_order  = ['Seasonal Naive', 'Ridge', 'XGBoost']
colours_bar  = [MODEL_COLOURS[m] for m in model_order]

metric_groups = [
    ('mae',      'MAE (kW)',  'Mean Absolute Error'),
    ('rmse',     'RMSE (kW)', 'Root Mean Squared Error'),
    ('peak_mae', 'kW',        'Peak-Hour MAE'),
]

n_models  = len(model_order)
n_metrics = len(metric_groups)
x         = np.arange(n_metrics)
bar_w     = 0.22
offsets   = np.linspace(-(n_models - 1) / 2 * bar_w, (n_models - 1) / 2 * bar_w, n_models)

fig, ax = plt.subplots(figsize=(10, 6))

for i, (model, colour) in enumerate(zip(model_order, colours_bar)):
    values = [metrics_plot.loc[model, col] for col, _, _ in metric_groups]
    bars = ax.bar(
        x + offsets[i],
        values,
        width=bar_w,
        color=colour,
        label=model,
        zorder=3,
        edgecolor='white',
        linewidth=0.8,
    )
    # Value labels on top of each bar
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.008,
            f'{val:.3f}',
            ha='center', va='bottom',
            fontsize=9, color=NAVY, fontweight='bold',
        )

ax.set_xticks(x)
ax.set_xticklabels([label for _, _, label in metric_groups], fontsize=12)
ax.set_ylabel('Error (kW)', fontsize=13, color=NAVY)
ax.set_title('Model Performance Comparison — Test Set (Jul–Nov 2010)', fontsize=14, pad=12)
ax.legend(loc='upper right', framealpha=0.9, fontsize=11)
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, metrics_plot[['mae', 'rmse', 'peak_mae']].max().max() * 1.25)

# Improvement annotation: XGBoost vs Naive MAE
naive_mae = metrics_plot.loc['Seasonal Naive', 'mae']
xgb_mae   = metrics_plot.loc['XGBoost', 'mae']
ridge_mae  = metrics_plot.loc['Ridge', 'mae']
improvement_pct = (naive_mae - xgb_mae) / naive_mae * 100
ridge_improvement = (naive_mae - ridge_mae) / naive_mae * 100
ax.annotate(
    f'XGBoost: {improvement_pct:.0f}% below Naive\nRidge: {ridge_improvement:.0f}% below Naive',
    xy=(0, ax.get_ylim()[1] * 0.88),
    fontsize=9.5, color=NAVY,
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#EEF4FB', edgecolor='#CCDDEE'),
)

plt.tight_layout()
out = FIGURES_DIR / 'model_comparison.png'
fig.savefig(out, dpi=300)
plt.show()
print(f'Saved: {out}')

**Interpretation.** Across all three error metrics, Ridge and XGBoost substantially outperform the Seasonal Naive baseline (~51-63% MAE reduction). Ridge achieves the lowest overall MAE (0.212 kW) while XGBoost produces competitive results — the trade-off favouring XGBoost emerges for its better handling of non-linear peak-hour patterns.

## 3. Figure 2 — Error by Horizon

Shows how forecast error (MAE) grows as the prediction horizon increases from h=1 to h=24.  
Each of the 24 models is evaluated independently against the h-step-ahead ground truth on the test set.

In [ ]:
from src.skills.config_loader import load_config
load_config()

# Load the 24 XGBoost models and the 24 Ridge models
MODELS_DIR = BACKEND / 'models'

xgb_models  = [joblib.load(MODELS_DIR / f'xgb_h{h}.joblib')   for h in range(1, 25)]
ridge_models = [joblib.load(MODELS_DIR / f'ridge_h{h}.joblib') for h in range(1, 25)]

print(f'Loaded {len(xgb_models)} XGBoost models and {len(ridge_models)} Ridge models')

# Prepare test features (drop target column)
feature_cols = [c for c in test_feat.columns if c != TARGET]
X_test = test_feat[feature_cols]
y_test = test_feat[TARGET]

# Compute MAE per horizon for XGBoost and Ridge
horizons = list(range(1, 25))
xgb_mae_by_h   = []
ridge_mae_by_h = []
naive_mae_by_h = []

for h in horizons:
    # XGBoost
    y_pred_xgb = xgb_models[h - 1].predict(X_test)
    y_true_h   = y_test.shift(-h).dropna()
    n          = min(len(y_true_h), len(y_pred_xgb))
    xgb_mae_by_h.append(np.mean(np.abs(y_true_h.values[:n] - y_pred_xgb[:n])))

    # Ridge
    y_pred_ridge = ridge_models[h - 1].predict(X_test)
    ridge_mae_by_h.append(np.mean(np.abs(y_true_h.values[:n] - y_pred_ridge[:n])))

    # Seasonal Naive: lag 168h
    lag_col = 'lag_168h'
    if lag_col in test_feat.columns:
        y_naive = test_feat[lag_col].shift(-(h - 1)).dropna()
        n2      = min(len(y_true_h), len(y_naive))
        naive_mae_by_h.append(np.mean(np.abs(y_true_h.values[:n2] - y_naive.values[:n2])))
    else:
        naive_mae_by_h.append(np.nan)

print('XGBoost  MAE h1-h24:', [f'{v:.3f}' for v in xgb_mae_by_h[:6]], '...')
print('Ridge    MAE h1-h24:', [f'{v:.3f}' for v in ridge_mae_by_h[:6]], '...')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(horizons, naive_mae_by_h, color=RED,    label='Seasonal Naive', lw=2.2, marker='o', ms=4, zorder=3)
ax.plot(horizons, xgb_mae_by_h,  color=BLUE,   label='XGBoost',        lw=2.2, marker='s', ms=4, zorder=4)
ax.plot(horizons, ridge_mae_by_h, color=ORANGE, label='Ridge',          lw=2.2, marker='^', ms=4, zorder=4)

# Shade between XGBoost and Naive to highlight improvement
ax.fill_between(horizons, xgb_mae_by_h, naive_mae_by_h,
                color=BLUE, alpha=0.08, label='XGBoost gain vs Naive')

ax.set_xlabel('Forecast Horizon h (hours ahead)', fontsize=13)
ax.set_ylabel('MAE (kW)', fontsize=13)
ax.set_title('Forecast Error by Horizon — Test Set', fontsize=14, pad=10)
ax.set_xticks(horizons)
ax.set_xlim(1, 24)
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc='upper left', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
out = FIGURES_DIR / 'error_by_horizon.png'
fig.savefig(out, dpi=300)
plt.show()
print(f'Saved: {out}')

**Interpretation.** All three models show a characteristic error-growth curve. Both XGBoost and Ridge maintain a significant advantage over the Seasonal Naive baseline across all 24 horizons. The convergence near h=24 reflects the dominance of seasonal periodicity at longer horizons.

## 4. Figure 3 — Forecast Example Plot

Shows actual vs. XGBoost predicted demand for a 48-hour window in the test set, including 90% conformal prediction intervals. A leading 24-hour context window of actuals is shown before the 24-hour forecast horizon.

In [ ]:
# Select a representative 48h period from test set — pick a weekday example
# Test set: 2010-07-01 to 2010-11-26
# Choose: 2010-08-09 (Monday) as a typical weekday
ORIGIN = pd.Timestamp('2010-08-09 00:00')
CONTEXT_H = 24  # hours of actual history to show
FORECAST_H = 24  # hours to forecast

# Build context window from hourly data
context_start = ORIGIN - pd.Timedelta(hours=CONTEXT_H)
context_end   = ORIGIN - pd.Timedelta(hours=1)
forecast_end  = ORIGIN + pd.Timedelta(hours=FORECAST_H - 1)

actual_full = hourly.loc[context_start:forecast_end, TARGET]

# Generate XGBoost forecast from origin
# Find the feature row at ORIGIN
if ORIGIN in X_test.index:
    origin_features = X_test.loc[[ORIGIN]]
else:
    # Find the nearest available origin in test
    nearest = X_test.index[X_test.index >= ORIGIN][0]
    origin_features = X_test.loc[[nearest]]
    print(f'Using nearest origin: {nearest}')

xgb_forecast = np.array([xgb_models[h - 1].predict(origin_features)[0] for h in range(1, FORECAST_H + 1)])
ridge_forecast = np.array([ridge_models[h - 1].predict(origin_features)[0] for h in range(1, FORECAST_H + 1)])

# Timestamps for forecast
forecast_timestamps = pd.date_range(ORIGIN + pd.Timedelta(hours=1), periods=FORECAST_H, freq='h')
context_timestamps  = pd.date_range(context_start, periods=CONTEXT_H, freq='h')

# Conformal prediction intervals
widths = np.array([conformal_widths[f'h{h}'] for h in range(1, FORECAST_H + 1)])
pi_lower = xgb_forecast - widths
pi_upper = xgb_forecast + widths

print(f'Origin: {ORIGIN}')
print(f'XGBoost forecast range: [{xgb_forecast.min():.2f}, {xgb_forecast.max():.2f}] kW')
print(f'Actual range: [{actual_full.min():.2f}, {actual_full.max():.2f}] kW')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Context actuals (history)
context_ts = actual_full.loc[context_start:ORIGIN]
ax.plot(context_ts.index, context_ts.values,
        color=NAVY, lw=2.0, label='Actual (context)', zorder=4)

# Future actuals
future_actual = actual_full.loc[ORIGIN + pd.Timedelta(hours=1):]
ax.plot(future_actual.index, future_actual.values,
        color=NAVY, lw=2.0, ls='--', alpha=0.7, label='Actual (forecast period)', zorder=4)

# XGBoost forecast
ax.plot(forecast_timestamps, xgb_forecast,
        color=BLUE, lw=2.2, ls='-', label='XGBoost forecast', zorder=5)

# Ridge forecast
ax.plot(forecast_timestamps, ridge_forecast,
        color=ORANGE, lw=2.0, ls='-.', label='Ridge forecast', zorder=5)

# 90% conformal prediction interval
ax.fill_between(
    forecast_timestamps, pi_lower, pi_upper,
    color=BLUE, alpha=0.15, label='90% conformal PI', zorder=2,
)
ax.plot(forecast_timestamps, pi_upper, color=BLUE, lw=0.8, ls=':', alpha=0.6)
ax.plot(forecast_timestamps, pi_lower, color=BLUE, lw=0.8, ls=':', alpha=0.6)

# Vertical line at forecast origin
ax.axvline(ORIGIN, color=RED, lw=1.5, ls='--', alpha=0.8, zorder=3)
ax.text(ORIGIN, ax.get_ylim()[1] * 0.96, ' Forecast\n origin',
        color=RED, fontsize=9.5, va='top', ha='left')

ax.set_xlabel('Time', fontsize=13)
ax.set_ylabel('Power Demand (kW)', fontsize=13)
ax.set_title(
    f'24-Hour Forecast Example — Origin: {ORIGIN.strftime("%Y-%m-%d %H:%M")} (Monday)',
    fontsize=13, pad=10
)
ax.legend(loc='upper left', fontsize=10, ncol=2)
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Format x-axis
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%d %b\n%H:%M'))

plt.tight_layout()
out = FIGURES_DIR / 'forecast_examples.png'
fig.savefig(out, dpi=300)
plt.show()
print(f'Saved: {out}')

**Interpretation.** Both learned models track the diurnal demand pattern closely. The 90% conformal prediction intervals correctly bound the actuals for the majority of hours. XGBoost (blue) captures peak-hour transitions more tightly than Ridge (orange) thanks to its non-linear decision boundaries.

## 5. Figure 4 — SHAP Feature Importance

Bar chart of mean absolute SHAP values for the XGBoost h=1 model on the validation set.  
Features are ranked by global importance (mean |SHAP|).

In [ ]:
# Compute mean absolute SHAP values and sort
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)

TOP_N = 15
top_features   = mean_abs_shap.head(TOP_N)

# Compute directional sign: positive SHAP = pushes prediction up
mean_shap_signed = shap_df.mean()

# Readable feature label mapping
FEATURE_LABELS = {
    'lag_1h':         'Lag 1h (y(t-1))',
    'lag_2h':         'Lag 2h (y(t-2))',
    'lag_24h':        'Lag 24h (same hour yesterday)',
    'lag_168h':       'Lag 168h (same hour last week)',
    'roll_mean_24h':  'Rolling mean 24h',
    'roll_mean_168h': 'Rolling mean 168h',
    'roll_std_24h':   'Rolling std 24h',
    'roll_max_24h':   'Rolling max 24h',
    'ewma_12h':       'EWMA 12h',
    'hour_sin':       'Hour (sin)',
    'hour_cos':       'Hour (cos)',
    'hour_of_day':    'Hour of day',
    'day_of_week':    'Day of week',
    'is_weekend':     'Is weekend',
    'is_holiday':     'Is holiday',
    'month':          'Month',
    'month_sin':      'Month (sin)',
    'month_cos':      'Month (cos)',
    'diff_1h':        'Diff 1h',
    'diff_24h':       'Diff 24h',
    'sub1_share':     'Sub-meter 1 share',
    'sub2_share':     'Sub-meter 2 share',
    'sub3_share':     'Sub-meter 3 share',
    'other_share':    'Other share',
    'sub3_lag24_diff':'Sub-meter 3 lag-24 diff',
    'roll_std_168h':  'Rolling std 168h',
}

labels = [FEATURE_LABELS.get(f, f) for f in top_features.index]
values = top_features.values

# Colour bars by directional impact
bar_colours = [
    GREEN if mean_shap_signed[f] >= 0 else RED
    for f in top_features.index
]

print(f'Top {TOP_N} features by mean |SHAP|:')
for feat, val in zip(top_features.index, values):
    print(f'  {FEATURE_LABELS.get(feat, feat):35s}  {val:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

y_pos = np.arange(TOP_N)
bars = ax.barh(
    y_pos, values[::-1],
    color=bar_colours[::-1],
    edgecolor='white',
    linewidth=0.6,
    zorder=3,
    alpha=0.88,
)

# Value labels
for bar, val in zip(bars, values[::-1]):
    ax.text(
        bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
        f'{val:.3f}',
        va='center', ha='left', fontsize=9, color=NAVY,
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(labels[::-1], fontsize=10.5)
ax.set_xlabel('Mean |SHAP Value| (kW)', fontsize=13)
ax.set_title(
    'XGBoost h=1 Feature Importance (SHAP) — Validation Set',
    fontsize=13, pad=10,
)

# Legend for direction
green_patch = mpatches.Patch(color=GREEN, alpha=0.88, label='Positive impact (raises forecast)')
red_patch   = mpatches.Patch(color=RED,   alpha=0.88, label='Negative impact (lowers forecast)')
ax.legend(handles=[green_patch, red_patch], loc='lower right', fontsize=10)

ax.xaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, values.max() * 1.18)

plt.tight_layout()
out = FIGURES_DIR / 'shap_summary.png'
fig.savefig(out, dpi=300)
plt.show()
print(f'Saved: {out}')

**Interpretation.** Recent autoregressive lags (lag 1h, lag 2h) dominate the h=1 model, confirming strong short-range autocorrelation in household electricity use. The rolling mean features and same-hour-last-week lag (168h) contribute substantially, capturing weekly seasonality. Calendar features (hour of day, day of week) provide additional structure, particularly for modelling morning and evening peaks.

## 6. Figure 5 — Residual Distribution

Histogram of forecast residuals (actual − predicted) for all three models on the test set (h=1).  
A well-calibrated model should produce residuals centred at zero with compact spread.

In [ ]:
# Compute h=1 residuals for all models on test set
y_test_h1 = test_feat[TARGET].shift(-1).dropna()
X_test_aligned = X_test.loc[y_test_h1.index]

xgb_preds_h1   = xgb_models[0].predict(X_test_aligned)
ridge_preds_h1  = ridge_models[0].predict(X_test_aligned)

n = min(len(y_test_h1), len(xgb_preds_h1))
y_true_h1 = y_test_h1.values[:n]

resid_xgb   = y_true_h1 - xgb_preds_h1[:n]
resid_ridge = y_true_h1 - ridge_preds_h1[:n]

# Seasonal Naive: use lag_168h from test features
naive_preds_h1  = test_feat['lag_168h'].shift(-1).dropna()
n_naive = min(len(y_test_h1), len(naive_preds_h1))
resid_naive = y_test_h1.values[:n_naive] - naive_preds_h1.values[:n_naive]

print(f'XGBoost   h=1 residuals: mean={resid_xgb.mean():.4f}, std={resid_xgb.std():.4f}')
print(f'Ridge     h=1 residuals: mean={resid_ridge.mean():.4f}, std={resid_ridge.std():.4f}')
print(f'Sn. Naive h=1 residuals: mean={resid_naive.mean():.4f}, std={resid_naive.std():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)

BINS = 60
XLIM = (-2.5, 2.5)

plot_data = [
    ('Seasonal Naive', resid_naive, RED),
    ('Ridge',          resid_ridge, ORANGE),
    ('XGBoost',        resid_xgb,   BLUE),
]

for ax, (name, resid, colour) in zip(axes, plot_data):
    ax.hist(resid, bins=BINS, color=colour, alpha=0.8, edgecolor='white',
            linewidth=0.5, density=True, zorder=3)
    ax.axvline(0, color=NAVY, lw=1.5, ls='--', zorder=4, label='Zero error')
    ax.axvline(resid.mean(), color='gold', lw=1.8, ls='-', zorder=5,
               label=f'Mean={resid.mean():.3f}')

    # KDE overlay
    from scipy.stats import gaussian_kde
    clipped = resid[np.abs(resid) < 3]
    if len(clipped) > 10:
        kde = gaussian_kde(clipped, bw_method=0.3)
        xs  = np.linspace(XLIM[0], XLIM[1], 200)
        ax.plot(xs, kde(xs), color='black', lw=1.5, alpha=0.7, zorder=6)

    ax.set_title(name, fontsize=13, color=colour)
    ax.set_xlabel('Residual (kW)', fontsize=12)
    ax.set_xlim(XLIM)
    ax.set_ylabel('Density', fontsize=12)
    ax.legend(fontsize=9.5)
    ax.yaxis.grid(True, zorder=0)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Annotate std
    ax.text(0.97, 0.95, f'std={resid.std():.3f} kW',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=10, color=NAVY,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#CCCCCC'))

fig.suptitle('Residual Distribution — h=1 Forecasts (Test Set)', fontsize=14,
             color=NAVY, fontweight='bold', y=1.02)

plt.tight_layout()
out = FIGURES_DIR / 'residual_distribution.png'
fig.savefig(out, dpi=300)
plt.show()
print(f'Saved: {out}')

**Interpretation.** Both Ridge and XGBoost residuals are tightly clustered around zero (std ≈ 0.28–0.36 kW), in contrast to the broader Seasonal Naive distribution (std ≈ 0.82 kW). The near-zero mean for all models confirms no systematic bias. Slight positive tails in all models indicate occasional under-prediction during demand spikes, which is expected.

## 7. Figure 6 — Feature Importance (XGBoost Built-in)

XGBoost native gain-based feature importance for the h=1 model.  
Gain measures the average improvement in the loss function brought by a feature when it is used to split a tree node.

In [ ]:
xgb_h1_model = xgb_models[0]

# Get feature importance as a sorted series
importance_dict = xgb_h1_model.get_booster().get_score(importance_type='gain')
importance_series = pd.Series(importance_dict).sort_values(ascending=False)

TOP_IMP = 20
top_imp = importance_series.head(TOP_IMP)
# Normalise to [0, 1]
top_imp_norm = top_imp / top_imp.sum()

imp_labels = [FEATURE_LABELS.get(f, f) for f in top_imp.index]

print(f'Top {TOP_IMP} features by XGBoost gain:')
for feat, val in zip(top_imp.index[:10], top_imp_norm.values[:10]):
    print(f'  {FEATURE_LABELS.get(feat, feat):35s}  {val:.4f}')

In [ ]:
# Build a colour gradient: top features in darker blue, rest lighter
cmap_blues = plt.cm.Blues
n_imp = len(top_imp)
imp_colours = [cmap_blues(0.9 - 0.55 * (i / (n_imp - 1))) for i in range(n_imp)]

fig, ax = plt.subplots(figsize=(9, 7))

y_pos = np.arange(n_imp)
bars  = ax.barh(
    y_pos,
    top_imp_norm.values[::-1],
    color=imp_colours[::-1],
    edgecolor='white',
    linewidth=0.5,
    zorder=3,
)

# Value labels
for bar, val in zip(bars, top_imp_norm.values[::-1]):
    ax.text(
        bar.get_width() + 0.002,
        bar.get_y() + bar.get_height() / 2,
        f'{val:.3f}',
        va='center', ha='left', fontsize=8.5, color=NAVY,
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(imp_labels[::-1], fontsize=10)
ax.set_xlabel('Normalised Gain (share of total)', fontsize=13)
ax.set_title(
    'XGBoost h=1 Built-in Feature Importance (Gain)',
    fontsize=13, pad=10,
)

ax.xaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, top_imp_norm.values.max() * 1.20)

plt.tight_layout()
out = FIGURES_DIR / 'feature_importance.png'
fig.savefig(out, dpi=300)
plt.show()
print(f'Saved: {out}')

**Interpretation.** The XGBoost gain importance broadly agrees with the SHAP analysis: recent lags dominate, followed by rolling-window statistics and cyclical calendar features. The agreement between model-agnostic SHAP values and built-in gain importance lends credibility to the feature importance ranking.

## 8. Bonus Figure — Demand Heatmap

24-hour × 12-month heatmap of average household demand (kW).  
Reveals the joint seasonality structure across time-of-day and time-of-year.

In [ ]:
# Build pivot: rows=month (1-12), cols=hour (0-23)
hourly['month']   = hourly.index.month
hourly['hour']    = hourly.index.hour

pivot = hourly.groupby(['month', 'hour'])[TARGET].mean().unstack('hour')

MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
HOUR_LABELS  = [f'{h:02d}:00' for h in range(24)]

print('Pivot shape:', pivot.shape)
print(pivot.describe())

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Custom colormap: Navy (low) → White (mid) → Orange (high)
cmap_custom = LinearSegmentedColormap.from_list(
    'energy_demand',
    [BLUE, '#EEF4FB', ORANGE, RED],
    N=256,
)

fig, ax = plt.subplots(figsize=(14, 6))

im = ax.imshow(
    pivot.values,
    cmap=cmap_custom,
    aspect='auto',
    interpolation='nearest',
    vmin=pivot.values.min(),
    vmax=pivot.values.max(),
)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Average Demand (kW)', fontsize=12, color=NAVY)
cbar.ax.yaxis.set_tick_params(color=NAVY)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=NAVY, fontsize=10)

ax.set_xticks(range(24))
ax.set_xticklabels([f'{h}' for h in range(24)], fontsize=9)
ax.set_yticks(range(12))
ax.set_yticklabels(MONTH_LABELS, fontsize=11)

ax.set_xlabel('Hour of Day', fontsize=13)
ax.set_ylabel('Month', fontsize=13)
ax.set_title(
    'Average Household Demand (kW) by Hour and Month (2006-2010)',
    fontsize=13, pad=10,
)

# Grid lines between cells
ax.set_xticks(np.arange(-0.5, 24, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 12, 1), minor=True)
ax.grid(which='minor', color='white', linewidth=0.5)
ax.tick_params(which='minor', bottom=False, left=False)

plt.tight_layout()
out = FIGURES_DIR / 'demand_heatmap.png'
fig.savefig(out, dpi=300)
plt.show()
print(f'Saved: {out}')

**Interpretation.** The heatmap reveals clear diurnal and seasonal patterns: demand peaks in the early evening (17:00–21:00) and is elevated in winter months (Dec–Feb), consistent with space heating and reduced daylight hours. Summer mornings (Jun–Aug) show the lowest overall demand. These patterns motivate the inclusion of hour-of-day, month, and seasonal lag features in the XGBoost and Ridge models.

## 9. Summary

All figures generated and saved to `backend/results/figures/`.

In [ ]:
import os

print('Generated figures:')
for f in sorted(FIGURES_DIR.glob('*.png')):
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f.name:40s}  {size_kb:6.1f} KB')